In [221]:
# Compare questions obtained from Maven database against the list of element obtained from the XML structure generated by Alex

In [222]:
import pandas as pd

In [223]:
df_qs = pd.read_csv('questions_package_model.csv',na_filter=False)
len(df_qs)

14985

In [224]:
# Remove questions beginning with DS and VF
df_qs = df_qs[~df_qs['QUESTION_ID'].str.startswith('DS')]
len(df_qs)

10724

In [225]:
df_qs = df_qs[~df_qs['QUESTION_ID'].str.startswith('VF')]
len(df_qs)

10456

In [226]:
df_qs = df_qs[~df_qs['QUESTION_ID'].str.startswith('CQFL')]
len(df_qs)

10128

In [227]:
df_qs = df_qs[~df_qs['QUESTION_ID'].str.startswith('CWFL')]
len(df_qs)

10073

In [228]:
df_qs = df_qs[~df_qs['QUESTION_ID'].str.startswith('KFLR')]
len(df_qs)

10066

In [229]:
df_qs[df_qs.duplicated(['QUESTION_ID','MODEL_NAME'],keep=False)]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
12772,TOC_VIS,UNKNOWN,CLINICAL,DiseaseSurveillanceModel_STD
13770,TOC_VIS,YES,SURRG_REINTERVIEW,DiseaseSurveillanceModel_STD


In [230]:
df_qs= df_qs.drop_duplicates(['QUESTION_ID','MODEL_NAME'])

In [231]:
df_qs = df_qs.reset_index(drop=True)
df_qs

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,11/7/2019,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
1,HAZARD_WATER_TAP_NO,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
2,AGE_FLAG_CHILD,CHILD,ADMINISTRATIVE,Childhood_Lead_Child_Model
3,HAZARD_WATER_ASSESSMENT_YES,YES,LEAD_HAZARDS,Childhood_Lead_Child_Model
4,CHECKLIST_VACCINATIONS_NO,NO,SCHOOL_READINESS,Childhood_Lead_Child_Model
...,...,...,...,...
10060,MEDICAL_LICENSE_NUMBER,9939,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10061,LAST_NAME,Zagorski,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10062,CURRENT_TELEPHONE_WORK,(808) 984-3493,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10063,SENTINEL_PROVIDER_NO,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [232]:
# Compare against xml parsed elements from Alex
df_xml = pd.read_csv('input/db_fields_alex.csv',na_filter=False)
len(df_xml)

16955

In [233]:
df_xml.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16955 entries, 0 to 16954
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Model             16955 non-null  object
 1   PackageID         16955 non-null  object
 2   UniqueID          16955 non-null  object
 3   ParentQuestionID  16955 non-null  object
 4   ChoiceValue       16955 non-null  object
 5   DisplayText       16955 non-null  object
dtypes: object(6)
memory usage: 794.9+ KB


In [234]:
df_xml.head()

,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText
0,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_ADULT,AGE_FLAG,ADULT,Adult Labs
1,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_CHILD,AGE_FLAG,CHILD,Child Labs
2,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_MIXED,AGE_FLAG,MIXED,Child and Adult Mixed Labs
3,Childhood_Lead_Child_Model,ADMINISTRATIVE,AGE_FLAG_UNKNOWN,AGE_FLAG,UNKNOWN,Missing Information
4,Childhood_Lead_Child_Model,ADMINISTRATIVE,ANY_OVER_5_OUTPUT,,,This Is a Case (>=5 Capillary or Venous):


In [235]:
# Compare UniqueID field against QUESTION_ID field in database

In [236]:
# Checking for dupes
df_xml[df_xml.duplicated(['Model','UniqueID'],keep=False)]

,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText


In [237]:
df_xml.loc[:,'Model'].value_counts(dropna=False)

Model
DiseaseSurveillanceModel_General        6847
DiseaseSurveillanceModel_STD            4012
DiseaseSurveillanceModel_Coinfection    1607
DiseaseSurveillanceModel_Hep            1501
ClusterModel                            1259
Childhood_Lead_Child_Model              1080
Childhood_Lead_Investigation_Model       393
EHRAndScreeningModel_STD                 139
PortalApplicationModel                    52
EmployerModel                             38
DiseaseSurveillanceModel_AggNETSS         20
IsolationAndControlModel                   7
Name: count, dtype: int64

In [238]:
df_qs.loc[:,'MODEL_NAME'].value_counts(dropna=False)

MODEL_NAME
DiseaseSurveillanceModel_General        5324
DiseaseSurveillanceModel_STD            2138
DiseaseSurveillanceModel_Hep            1210
Childhood_Lead_Child_Model               570
ClusterModel                             409
Childhood_Lead_Investigation_Model       196
DiseaseSurveillanceModel_Coinfection     121
PortalApplicationModel                    51
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         17
IsolationAndControlModel                   7
Name: count, dtype: int64

In [239]:
# merge the two datasets on df_qs[question_id,model_name] and df_xml[unique_id,model]
df_merge = pd.merge(df_qs,df_xml,left_on=['MODEL_NAME','QUESTION_ID'],right_on=['Model','UniqueID'],how='outer',indicator=True)
len(df_merge)

17003

In [240]:
df_qs_only = df_merge[df_merge['_merge']=='left_only']
len(df_qs_only)

48

In [241]:
df_xml_only = df_merge[df_merge['_merge']=='right_only']
len(df_xml_only)

6938

In [242]:
df_both = df_merge[df_merge['_merge']=='both']
len(df_both)

10017

In [243]:
df_qs_only.to_csv('output/elements_db_only_may1.csv',index=False)

In [244]:
df_both.to_csv('output/elements_both_db_and_xml_may1.csv',index=False)

In [245]:
df_xml_only.to_csv('output/elements_xml_only_may1.csv',index=False)

In [246]:
df_xml_only.columns

Index(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME', 'Model',
       'PackageID', 'UniqueID', 'ParentQuestionID', 'ChoiceValue',
       'DisplayText', '_merge'],
      dtype='object')

In [247]:
# Drop blanks and remove unnecessary columns
df_xml_only = df_xml_only.drop(['QUESTION_ID', 'value', 'QUESTIONSET_ID', 'MODEL_NAME','_merge'],axis=1)

In [248]:
len(df_xml_only)

6938

In [249]:
df_xml_only[df_xml_only.duplicated(['Model','ParentQuestionID'])]

,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText
22,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_CURRENT_ISLAND,,,Island
23,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT,,,Date of contact:
25,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT_DATE_METHOD_FAX,ADULT_DATE_OF_CONTACT_DATE_METHOD,FAX,Fax
26,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT_DATE_METHOD_IN_PERSON,ADULT_DATE_OF_CONTACT_DATE_METHOD,IN_PERSON,In person
27,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT_DATE_METHOD_LETTER,ADULT_DATE_OF_CONTACT_DATE_METHOD,LETTER,Letter
...,...,...,...,...,...,...
16939,EmployerModel,LAB_EMPLOYER,TELEPHONE_CELL,,,Cell phone
16940,EmployerModel,LAB_EMPLOYER,TELEPHONE_HOME,,,Home phone
16941,EmployerModel,LAB_EMPLOYER,TELEPHONE_WORK,,,Phone
16942,EmployerModel,LAB_EMPLOYER,WORK_EXTENSION,,,Extension


In [250]:
df_xml_only = df_xml_only.drop_duplicates(['Model','ParentQuestionID'])

In [251]:
len(df_xml_only)

1355

In [252]:
#df_xml_only = df_xml_only[df_xml_only['ParentQuestionID']!='']

In [253]:
df_xml_only

,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText
21,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_COC,,,COC Code:
24,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT_DATE_METHOD_EMAIL,ADULT_DATE_OF_CONTACT_DATE_METHOD,EMAIL,Email
31,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_DATE_OF_CONTACT_DATE_TYPE_LAB,ADULT_DATE_OF_CONTACT_DATE_TYPE,LAB,Lab
41,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_HISPANIC_NO,ADULT_HISPANIC,NO,No
45,Childhood_Lead_Child_Model,ADULT_LEAD,ADULT_RACE_AMERICAN_INDIAN_ALASKAN_NATIVE,ADULT_RACE,AMERICAN_INDIAN_ALASKAN_NATIVE,American Indian Alaskan Native
...,...,...,...,...,...,...
16906,EmployerModel,LAB_EMPLOYER,BUSINESS_TYPE,,,Business type
16907,EmployerModel,LAB_EMPLOYER,CASE_TYPE_EMP,CASE_TYPE,EMP,Employer
16913,EmployerModel,LAB_EMPLOYER,CONTACT_METHOD_1,CONTACT_METHOD,1,Mail
16927,EmployerModel,LAB_EMPLOYER,LAB_TYPE_EMP,LAB_TYPE,EMP,Employer


In [254]:
df_xml_only.to_csv('output/elements_xml_only_may1_dupes_removed.csv',index=False)

In [255]:
len(df_both)

10017

In [256]:
df_both

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME,Model,PackageID,UniqueID,ParentQuestionID,ChoiceValue,DisplayText,_merge
0,ACTIVITIES_FAMILY_LEAD_TESTED_BLL,7.8,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model,Childhood_Lead_Child_Model,HOUSEKEEPING_PRACTICES,ACTIVITIES_FAMILY_LEAD_TESTED_BLL,,,Blood Lead Level,both
1,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,11/7/2019,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model,Childhood_Lead_Child_Model,HOUSEKEEPING_PRACTICES,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,,,Date of Test,both
2,ACTIVITIES_FAMILY_LEAD_TESTED_FIRST_NAME,unknown,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model,Childhood_Lead_Child_Model,HOUSEKEEPING_PRACTICES,ACTIVITIES_FAMILY_LEAD_TESTED_FIRST_NAME,,,First Name,both
3,ACTIVITIES_FAMILY_LEAD_TESTED_LAST_NAME,Yacapin,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model,Childhood_Lead_Child_Model,HOUSEKEEPING_PRACTICES,ACTIVITIES_FAMILY_LEAD_TESTED_LAST_NAME,,,Last Name,both
4,ACTIVITIES_FAMILY_LEAD_TESTED_NO,NO,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model,Childhood_Lead_Child_Model,HOUSEKEEPING_PRACTICES,ACTIVITIES_FAMILY_LEAD_TESTED_NO,ACTIVITIES_FAMILY_LEAD_TESTED,NO,No,both
...,...,...,...,...,...,...,...,...,...,...,...
16998,TYPE_OF_USER_NO,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_NO,TYPE_OF_USER,NO,No,both
16999,TYPE_OF_USER_STD_NO,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_STD_NO,TYPE_OF_USER_STD,NO,No,both
17000,TYPE_OF_USER_STD_YES,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_STD_YES,TYPE_OF_USER_STD,YES,Yes,both
17001,TYPE_OF_USER_YES,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel,PortalApplicationModel,DEMOGRAPHIC_AND_PROFESSIONAL,TYPE_OF_USER_YES,TYPE_OF_USER,YES,Yes,both


In [257]:
df_both= df_both[['QUESTION_ID','value','QUESTIONSET_ID','MODEL_NAME']]

In [258]:
def join_all_except_last(question,answer):

    # Check if answer matches last part of the question
    start=len(question)-len(answer)
    if(question[start:]==answer):
        return question[:start].strip('_')
    else:
        return question

In [259]:
# Truncate question if answer is contained.
# If last part of question equals value then truncate the question
df_both.loc[:,'QUESTION_ID']  = [ join_all_except_last(q,a) for q,a in zip(df_both.loc[:,'QUESTION_ID'],df_both.loc[:,'value'])]

In [260]:
df_both[df_both.duplicated(['QUESTION_ID','MODEL_NAME'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
6,ACTIVITIES_FAMILY_LEAD_TESTED_SIBLINGS,YES,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
8,ACTIVITIES_FAMILY_LEAD_TESTED,YES,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
11,ACTIVITIES_FAMILY_RISK_MEMBER_EXPOSURE_TYPE,OCCUPATIONAL,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
19,ACTIVITIES_FAMILY_RISK,YES,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
56,AGE_FLAG,CHILD,ADMINISTRATIVE,Childhood_Lead_Child_Model
...,...,...,...,...
16986,REPORTS_ON_BEHALF_OF_AGENCY,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
16994,SENTINEL_PROVIDER,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
16997,STD_SCREENING_PARTNER,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
17000,TYPE_OF_USER_STD,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [261]:
df_both = df_both.drop_duplicates(['QUESTION_ID','MODEL_NAME'])

In [262]:
df_both = df_both.reset_index(drop=True)

In [263]:
len(df_both)

6205

In [264]:
df_both

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,ACTIVITIES_FAMILY_LEAD_TESTED_BLL,7.8,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
1,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,11/7/2019,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
2,ACTIVITIES_FAMILY_LEAD_TESTED_FIRST_NAME,unknown,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
3,ACTIVITIES_FAMILY_LEAD_TESTED_LAST_NAME,Yacapin,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
4,ACTIVITIES_FAMILY_LEAD_TESTED,NO,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
...,...,...,...,...
6200,STD_SCREENING_PARTNER_ID,66888,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6201,STD_SCREENING_PARTNER,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6202,TYPE_OF_USER,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6203,TYPE_OF_USER_STD,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [265]:
df_both.value_counts(['MODEL_NAME'],dropna=False)

MODEL_NAME                          
DiseaseSurveillanceModel_General        3166
DiseaseSurveillanceModel_STD            1361
DiseaseSurveillanceModel_Hep             754
Childhood_Lead_Child_Model               386
ClusterModel                             204
Childhood_Lead_Investigation_Model       136
DiseaseSurveillanceModel_Coinfection     114
PortalApplicationModel                    41
EHRAndScreeningModel_STD                  22
DiseaseSurveillanceModel_AggNETSS         14
IsolationAndControlModel                   7
Name: count, dtype: int64

In [266]:
df_both.to_csv('output/elements_both_db_and_xml_dupes_removed.csv',index=False)

In [267]:
# Collapsing the 10K database elements - reducing unique questions to parent questions.
# Irrespective of whether these elements match with those obtained from XML parsing
df_qs

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,11/7/2019,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
1,HAZARD_WATER_TAP_NO,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
2,AGE_FLAG_CHILD,CHILD,ADMINISTRATIVE,Childhood_Lead_Child_Model
3,HAZARD_WATER_ASSESSMENT_YES,YES,LEAD_HAZARDS,Childhood_Lead_Child_Model
4,CHECKLIST_VACCINATIONS_NO,NO,SCHOOL_READINESS,Childhood_Lead_Child_Model
...,...,...,...,...
10060,MEDICAL_LICENSE_NUMBER,9939,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10061,LAST_NAME,Zagorski,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10062,CURRENT_TELEPHONE_WORK,(808) 984-3493,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10063,SENTINEL_PROVIDER_NO,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [268]:
df_qs.loc[:,'QUESTION_ID'] = [ join_all_except_last(q,a) for q,a in zip(df_qs.loc[:,'QUESTION_ID'],df_qs.loc[:,'value'])]

In [269]:
df_qs[df_qs.duplicated(['QUESTION_ID','MODEL_NAME'])]

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
16,AGE_FLAG,UNKNOWN,ADMINISTRATIVE,Childhood_Lead_Child_Model
24,HAZARD_SOIL_ACTION_SAMPLES_NO_WHY,NO_BARE_SOIL,LEAD_HAZARDS,Childhood_Lead_Child_Model
26,HAZARD_WATER_ASSESSMENT,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
37,HAZARD_BEHAV_ASSESSMENT_MOUTHING,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
61,HAZARD_SOIL_ACTION_SAMPLES,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
...,...,...,...,...
10050,LICENSED_IN_HAWAII,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10051,TYPE_OF_USER,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10059,ACCOUNT_SUBMISSION_PREPROCESSED,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
10063,SENTINEL_PROVIDER,NO,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [270]:
df_qs = df_qs.drop_duplicates(['QUESTION_ID','MODEL_NAME'])

In [271]:
df_qs = df_qs.reset_index(drop=True)

In [272]:
df_qs

,QUESTION_ID,value,QUESTIONSET_ID,MODEL_NAME
0,ACTIVITIES_FAMILY_LEAD_TESTED_DATE,11/7/2019,HOUSEKEEPING_PRACTICES,Childhood_Lead_Child_Model
1,HAZARD_WATER_TAP,NO,LEAD_HAZARDS,Childhood_Lead_Child_Model
2,AGE_FLAG,CHILD,ADMINISTRATIVE,Childhood_Lead_Child_Model
3,HAZARD_WATER_ASSESSMENT,YES,LEAD_HAZARDS,Childhood_Lead_Child_Model
4,CHECKLIST_VACCINATIONS,NO,SCHOOL_READINESS,Childhood_Lead_Child_Model
...,...,...,...,...
6227,ACCOUNT_SUBMISSION_PREPROCESSED,YES,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6228,CURRENT_TELEPHONE_FAX,(909) 597-2207,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6229,MEDICAL_LICENSE_NUMBER,9939,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel
6230,LAST_NAME,Zagorski,DEMOGRAPHIC_AND_PROFESSIONAL,PortalApplicationModel


In [273]:
df_qs.to_csv("output/all_db_elements_use_this.csv",index=False)